# PEMFC Cathode -- $\tau=1.1$, $L_x=0.2$mm: Minus-Direction Time-Series Comparison

**Lightweight, standalone plotting notebook -- no FEniCSx needed.** Reads whatever
`minusdir_fields_frac*_idx*.npz` results already exist in Drive (from
`PEMFC_Tau1.1_Lx0.2mm_MinusDirection_Fields.ipynb`) and plots the full $I(t)$ and $s_{max}(t)$
trajectories for every available (fraction, eigenvector index) combination, to compare how the
relaxation behavior changes across the fractions tested -- some settle to a different steady
state, others may still be relaxing or ringing at the end of the run.

**Now covers all three unstable eigenvector directions (`idx=0,1,2`)**, not just a single
direction like the original version of this notebook did. One row per index, so the three
directions' relaxation behavior can be compared side by side; each row shows every completed
fraction for that index the same way the original single-direction plot did (same outlier
filter, same viridis-by-fraction coloring).

**Also now draws a second reference line: the empirical "lower branch"** (lower-$s_{max}$,
higher-$I$) at this same $\eta$. Continuation only ever traced ONE branch -- the original,
higher-$s_{max}$ one that becomes unstable past the fold and is drawn as the existing "original
steady" dashed line -- and it stalls at the fold with no way to reach any other branch that
might coexist at the same $\eta$. It never independently traced a second branch. This notebook
instead estimates that second branch's location the only way currently available: by averaging
the final state of every (fraction, idx) run that this project's own tail-window heuristic (see
`MinusDirection_Summary.ipynb`) classifies as "settled -- new steady state" (i.e. flat tail,
clearly displaced from the original). Runs still classified "still relaxing" are excluded from
the average and reported separately, since several of them (especially near the fold, where
dynamics genuinely slow down -- "critical slowing down") may still be creeping toward this same
attractor rather than having reached it -- so treat this line as a provisional, likely-still-
approaching estimate, not a converged fixed point.

For a systematic table + summary plots across the *whole* sweep instead of full time-series
detail, see `PEMFC_Tau1.1_Lx0.2mm_MinusDirection_Summary.ipynb`.


## Mount Google Drive

In [ ]:
import os
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Google Drive already mounted')

DRIVE_BACKUP = "/content/drive/MyDrive/pemfc_tau1.1_Lx0.2mm_minusdir_fields_backup"
print(f"Reading results from: {DRIVE_BACKUP}")
assert os.path.isdir(DRIVE_BACKUP), f"{DRIVE_BACKUP} not found -- run the Fields notebook first."


## Plot: $I(t)$ and $s_{max}(t)$ across all available fractions, one row per eigenvector index

In [ ]:
import os
import re
import glob
import numpy as np
import matplotlib.pyplot as plt

# Only FINAL (complete) results -- explicitly exclude any still-in-progress
# checkpoint files, which have a different (partial, still-evolving) structure.
files = sorted(f for f in glob.glob(f"{DRIVE_BACKUP}/minusdir_fields_frac*_idx*.npz")
                if "_inprogress" not in f)
assert files, f"No completed results found in {DRIVE_BACKUP}."

# Parse each file's (fraction, eigenvector index) from its filename
# (frac0p00025_idx1.npz -> fraction=0.00025, idx=1) -- both are baked into the filename
# since the Fields notebook loops EIGVEC_INDICES_TO_TRY=[0,1,2] for every fraction.
FNAME_RE = re.compile(r"minusdir_fields_frac(\d+p\d+)_idx(\d+)\.npz$")


def parse_filename(fp):
    m = FNAME_RE.search(os.path.basename(fp))
    assert m, f"Unrecognized filename pattern: {fp}"
    fraction = float(m.group(1).replace("p", "."))
    idx = int(m.group(2))
    return fraction, idx


def flag_outliers(y, rel_thresh=0.3):
    """Marks a point as an outlier if it deviates from the AVERAGE of its two
    immediate neighbors by more than rel_thresh (default 30%) of that average.
    Confirmed the hard way that a sharp flooding front can produce a single-step
    L2-projection overshoot (a brief, spurious dip that recovers immediately the
    very next step) -- this is a simple neighbor-comparison filter for exactly
    that pattern, not a general-purpose outlier detector. First/last points are
    never flagged (no two-sided neighbor comparison possible)."""
    n = len(y)
    is_outlier = np.zeros(n, dtype=bool)
    for i in range(1, n - 1):
        neighbor_avg = 0.5 * (y[i - 1] + y[i + 1])
        if abs(y[i] - neighbor_avg) > rel_thresh * abs(neighbor_avg):
            is_outlier[i] = True
    return is_outlier


# ============ EDIT THESE (must match MinusDirection_Summary.ipynb's own thresholds --
# kept in sync by hand since this notebook is deliberately standalone/no shared import) ============
TAIL_WINDOW_S = 10.0       # how much of the END of each run's own time series counts as "tail"
TAIL_AMP_THRESH = 0.005    # s_max units -- tail range above this = "not flat"
MIN_SIGN_CHANGES = 2       # slope reversals within the tail needed to call it oscillating
NEAR_ORIGINAL_TOL = 0.02   # s_max units -- how close to the ORIGINAL steady counts as "returned"
# =====================================================================================


def classify_run(t, s_max, s_steady):
    """Same coarse settle/drift/oscillate classification as MinusDirection_Summary.ipynb,
    used here only to decide which runs' FINAL state is trustworthy enough to average into
    the empirical "lower branch" (second, previously-untraced steady-state family) reference
    line -- see the markdown cell above for the full rationale."""
    tail_mask = t >= (t[-1] - TAIL_WINDOW_S)
    s_tail = s_max[tail_mask]
    tail_amp = float(s_tail.max() - s_tail.min()) if len(s_tail) > 1 else 0.0
    ds = np.diff(s_tail)
    ds = ds[np.abs(ds) > 1e-6]  # ignore near-zero solver noise between accepted steps
    sign_changes = int(np.sum(np.diff(np.sign(ds)) != 0)) if len(ds) > 1 else 0
    near_original = abs(float(s_max[-1]) - s_steady) < NEAR_ORIGINAL_TOL
    if tail_amp > TAIL_AMP_THRESH:
        label = "oscillating / limit cycle" if sign_changes >= MIN_SIGN_CHANGES else "still relaxing"
    else:
        label = "settled -- back near original" if near_original else "settled -- new steady state"
    return label, tail_amp


entries = sorted((parse_filename(fp) + (fp,) for fp in files), key=lambda x: (x[1], x[0]))
by_idx = {}
for fraction, idx, fp in entries:
    by_idx.setdefault(idx, []).append((fraction, fp))

EIGVEC_INDICES = sorted(by_idx.keys())
print(f"Found {len(entries)} completed (fraction, idx) result(s) across "
      f"{len(EIGVEC_INDICES)} eigenvector index/indices: {EIGVEC_INDICES}")
for idx in EIGVEC_INDICES:
    fracs = [f for f, _ in by_idx[idx]]
    print(f"  idx={idx}: {len(fracs)} fraction(s) -- {fracs}")

# ---- First pass: classify every run's FINAL state, to build the empirical lower-branch
# reference (mean I_final/s_final over runs classified "settled -- new steady state" only).
# This is a separate, lightweight pass over the same files -- kept apart from the main
# plotting loop below so the plotting loop's own per-fraction data loading is unaffected.
new_branch_points, still_relaxing_points = [], []
I_steady_probe = s_steady_probe = None
for fraction, idx, fp in entries:
    d = np.load(fp)
    t, I_avg_raw, s_max_raw = d["t"], d["I_avg"] / 1e4, d["s_max"]
    if I_steady_probe is None:
        I_steady_probe = float(d["I_steady"]) / 1e4
        s_steady_probe = float(d["s_steady"])
    label, tail_amp = classify_run(t, s_max_raw, s_steady_probe)
    point = dict(idx=idx, fraction=fraction, I_final=float(I_avg_raw[-1]),
                 s_final=float(s_max_raw[-1]), t_end=float(t[-1]), tail_amp=tail_amp)
    if label == "settled -- new steady state":
        new_branch_points.append(point)
    elif label in ("still relaxing", "oscillating / limit cycle"):
        still_relaxing_points.append(point)

I_lower = s_lower = None
if new_branch_points:
    I_lower = float(np.mean([p["I_final"] for p in new_branch_points]))
    s_lower = float(np.mean([p["s_final"] for p in new_branch_points]))
    I_lower_std = float(np.std([p["I_final"] for p in new_branch_points]))
    s_lower_std = float(np.std([p["s_final"] for p in new_branch_points]))
    print(f"\nEmpirical lower branch (mean over {len(new_branch_points)} 'settled -- new "
          f"steady state' run(s)): I={I_lower:.4f}+-{I_lower_std:.4f} A/cm^2, "
          f"s_max={s_lower:.4f}+-{s_lower_std:.4f}")
    # Longest elapsed simulated time, not lowest tail amplitude, is the better "most trustworthy"
    # criterion here: near a fold, dynamics can crawl slowly enough that a 10s tail window looks
    # locally flat while the state is still creeping toward the true attractor (see caveat above)
    # -- more total simulated time is the stronger evidence of genuine convergence.
    best = max(new_branch_points, key=lambda p: p["t_end"])
    print(f"  Longest-duration point (t_end={best['t_end']:.1f}s, tail amplitude={best['tail_amp']:.5f}): "
          f"idx={best['idx']} frac={best['fraction']:g} -> I={best['I_final']:.4f}, s_max={best['s_final']:.4f}")
    if still_relaxing_points:
        print(f"  Excluded as still relaxing / not yet flat ({len(still_relaxing_points)} run(s)): " +
              ", ".join(f"idx={p['idx']} frac={p['fraction']:g} (s_max={p['s_final']:.3f})"
                        for p in still_relaxing_points) +
              " -- these may still be approaching the same attractor; if so, the true lower "
              "branch likely has even higher I / lower s_max than the mean above.")
else:
    print("\nNo runs yet classified 'settled -- new steady state' -- skipping the empirical "
          "lower-branch reference line.")

fig, axes = plt.subplots(len(EIGVEC_INDICES), 2, figsize=(12, 4.3 * len(EIGVEC_INDICES)),
                          squeeze=False)

I_steady = s_steady = eta_val = None
for row, idx in enumerate(EIGVEC_INDICES):
    frac_entries = sorted(by_idx[idx], key=lambda x: x[0])
    cmap = plt.cm.viridis(np.linspace(0, 1, len(frac_entries)))
    ax_I, ax_s = axes[row]

    for (fraction, fp), color in zip(frac_entries, cmap):
        d = np.load(fp)
        t, I_avg, s_max = d["t"], d["I_avg"] / 1e4, d["s_max"]
        if I_steady is None:
            I_steady, s_steady = float(d["I_steady"]) / 1e4, float(d["s_steady"])
            eta_val = float(d["eta"])

        I_outliers = flag_outliers(I_avg)
        if I_outliers.any():
            print(f"  idx={idx} frac={fraction:g}: filtered {I_outliers.sum()} outlier point(s) "
                  f"from I(t) at t={t[I_outliers].round(2).tolist()}")
        s_outliers = flag_outliers(s_max)
        if s_outliers.any():
            print(f"  idx={idx} frac={fraction:g}: filtered {s_outliers.sum()} outlier point(s) "
                  f"from s_max(t) at t={t[s_outliers].round(2).tolist()}")
        # Break the line at outliers (NaN) rather than dropping the point entirely --
        # keeps the time axis intact and avoids connecting across the gap with a
        # straight line that implies data that isn't there.
        I_clean = np.where(I_outliers, np.nan, I_avg)
        s_clean = np.where(s_outliers, np.nan, s_max)

        ax_I.plot(t, I_clean, "-", color=color, lw=1.3, label=f"frac={fraction:g}")
        if I_outliers.any():
            ax_I.plot(t[I_outliers], I_avg[I_outliers], "x", color=color, ms=6, mew=1.5)
        ax_s.plot(t, s_clean, "-", color=color, lw=1.3, label=f"frac={fraction:g}")
        if s_outliers.any():
            ax_s.plot(t[s_outliers], s_max[s_outliers], "x", color=color, ms=6, mew=1.5)

    ax_I.axhline(I_steady, color="gray", ls="--", lw=0.8, label="original steady")
    if I_lower is not None:
        ax_I.axhline(I_lower, color="#c0392b", ls="--", lw=0.8, label="lower branch (empirical)")
    ax_I.set_xlabel("time (s)")
    ax_I.set_ylabel(r"$I$ (A/cm$^2$)")
    ax_I.set_title(f"idx={idx}: current density relaxation" +
                    (f" (eta={eta_val:.4f})" if row == 0 and eta_val else ""))
    ax_I.legend(fontsize=7, ncol=2)
    ax_I.grid(alpha=0.3)

    ax_s.axhline(s_steady, color="gray", ls="--", lw=0.8, label="original steady")
    if s_lower is not None:
        ax_s.axhline(s_lower, color="#c0392b", ls="--", lw=0.8, label="lower branch (empirical)")
    ax_s.set_xlabel("time (s)")
    ax_s.set_ylabel(r"$s_{max}$")
    ax_s.set_title(f"idx={idx}: saturation relaxation")
    ax_s.legend(fontsize=7, ncol=2)
    ax_s.grid(alpha=0.3)

fig.suptitle("x markers = filtered single-step artifacts", fontsize=9, y=1.005)
plt.tight_layout()
out_png = f"{DRIVE_BACKUP}/minusdir_timeseries_comparison_all_idx.png"
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved to {out_png}")
